# Task 3 of 3 (`Verify_Output`)
### Lakeflow Jobs Orchestration Lab · CDC + Medallion Architecture
**Databricks Free Edition (serverless)**

This is the **third and final task** of the Job. It **explores and validates** the processing results by displaying the contents of the Silver and Gold tables.

**Depends on:** `Process_Medallion` (Task 2).

In [0]:
# Shared configuration (identical across the 3 Job tasks)
catalog = "workspace"
schema  = "medallion_cdc_orq"
volume  = "raw_data"

base_path = f"/Volumes/{catalog}/{schema}/{volume}"
landing   = f"{base_path}/landing"     # CDC feed events are landed here

# Ensure the required structure exists (idempotent)
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA  IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME  IF NOT EXISTS {catalog}.{schema}.{volume}")
spark.sql(f"USE {catalog}.{schema}")

print("catalog/schema :", f"{catalog}.{schema}")
print("landing        :", landing)

catalog/schema : workspace.medallion_cdc_orq
landing        : /Volumes/workspace/medallion_cdc_orq/raw_data/landing


## Silver (estado actual de los pedidos)

In [0]:
%sql
SELECT order_id, customer, product, amount, status
FROM   silver_orders
ORDER  BY order_id

order_id,customer,product,amount,status
1,Ana,Sneakers,90.0,delivered
3,Carla,Cap,15.0,pending
4,Diana,Hoodie,60.0,shipped
5,Elena,Scarf,20.0,pending


## Silver (Current State of the Orders)

In [0]:
%sql
SELECT order_id, customer, product, amount, status
FROM silver_orders
ORDER BY order_id;

order_id,customer,product,amount,status
1,Ana,Sneakers,90.0,delivered
3,Carla,Cap,15.0,pending
4,Diana,Hoodie,60.0,shipped
5,Elena,Scarf,20.0,pending


## Simple Quality Check

A verification task typically includes **assertions**. If something is not as expected, the task **fails**, and the Job notifies you. Here, we verify that the Gold table is not empty.

In [0]:
# Validation: Gold must contain at least one row
num = spark.table("gold_order_summary").count()
print(f"Gold contains {num} metric row(s).")

assert num > 0, "Gold is empty: the pipeline did not produce any results."

print("✅ Verification passed: the CDC + Medallion workflow executed successfully.")

Gold contains 3 metric row(s).
✅ Verification passed: the CDC + Medallion workflow executed successfully.


---
## Trigger an Error to Practice *Repair Run*

Uncomment the following cell and **run the Job again**. It will query a table that does not exist, causing **Task 3** to fail with a `TABLE_OR_VIEW_NOT_FOUND` error. In the Job run UI, you will see the task marked in red. Then, use **Repair run** to re-execute **only this task** (without rerunning Tasks 1 and 2) after fixing the error.

In [0]:
# Uncomment to simulate a failure and practice "Repair run"
# spark.sql("SELECT * FROM table_that_does_not_exist_usa").show()

### How to Repair It
1. Run the Job with the line above **uncommented** → **Task 3** fails.
2. In **Jobs & Pipelines → Runs**, open the failed run and review the error.
3. **Comment the line again** (fixing the "bug").
4. In the failed run, click **Repair run** → only `Verify_Output` is re-executed.
5. The run completes successfully: you recovered the workflow without rerunning the entire pipeline.